# SECTION 0 — ADD ROOT PATH

In [61]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent

sys.path.append(
    str(PROJECT_ROOT)
)

print(PROJECT_ROOT)

D:\Bách khoa\Intelligence System\Weather Forecast Dashboard UI\backend\ml_pipeline


# SECTION 1 — Import

In [62]:
from pathlib import Path

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from utils.data_loader import (
    DATA_PROVINCES,
    load_province_data
)

from utils.feature_engineering import (
    prepare_train_test_data,
)

from utils.evaluation import (
    evaluate_model,
    print_report,
    create_result_row,
    create_results_dataframe,
    rank_models
)

from utils.save_model import (
    save_complete_model_package
)

# SECTION 2 — Config

In [63]:
LOOKBACKS = [3, 5, 7, 10, 15, 30]

FORECAST_HORIZONS = [7]

ALL_RESULTS = []

# SECTION 3 — LOAD ONE PROVINCE

In [64]:
province_name = "Ha_Noi"

df = load_province_data(
    province_name
)

print(df.head())
print(df.shape)

        date destination  latitude  longitude  temp  humidity  wind_speed  \
0 2020-01-01      Ha_Noi   21.0285   105.8542  20.0        88         9.2   
1 2020-01-02      Ha_Noi   21.0285   105.8542  21.2        84        15.0   
2 2020-01-03      Ha_Noi   21.0285   105.8542  21.5        87        16.3   
3 2020-01-04      Ha_Noi   21.0285   105.8542  21.2        88        18.1   
4 2020-01-05      Ha_Noi   21.0285   105.8542  21.5        86        18.7   

   cloud_cover  pressure  rain  
0           83    1021.4     1  
1           75    1019.8     1  
2           82    1017.9     1  
3           77    1016.0     1  
4           80    1014.8     1  
(2324, 10)


# SECTION 4 — PREPARE DATA

In [65]:
lookback = 7
forecast_horizon = 1

(
    X_train,
    X_test,
    y_train,
    y_test,
    train_df,
    test_df
) = prepare_train_test_data(
    df=df,
    lookback=lookback,
    forecast_horizon=forecast_horizon
)

print(X_train.shape)
print(X_test.shape)

(2185, 35)
(131, 35)


# SECTION 5 — CREATE MODEL

In [66]:
model = Pipeline([

    (
        "scaler",
        StandardScaler()
    ),

    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced"
        )
    )
])

# SECTION 6 — TRAIN MODEL

In [67]:
model.fit(
    X_train,
    y_train
)

print("Training completed.")

Training completed.


# SECTION 7 — PREDICT

In [68]:
y_pred = model.predict(
    X_test
)

y_prob = model.predict_proba(
    X_test
)[:, 1]

# SECTION 8 — EVALUATE

In [69]:
metrics = evaluate_model(
    y_test,
    y_pred,
    y_prob
)

print(metrics)

print_report(
    y_test,
    y_pred
)

{'accuracy': 0.6641221374045801, 'precision': 0.8421052631578947, 'recall': 0.6666666666666666, 'f1': 0.7441860465116279, 'roc_auc': 0.7092261904761904}

Classification Report

              precision    recall  f1-score   support

         0.0       0.42      0.66      0.51        35
         1.0       0.84      0.67      0.74        96

    accuracy                           0.66       131
   macro avg       0.63      0.66      0.63       131
weighted avg       0.73      0.66      0.68       131



# SECTION 9 — SAVE SINGLE RESULT

In [70]:
result_row = create_result_row(
    province=province_name,
    lookback=lookback,
    forecast_horizon=forecast_horizon,
    metrics=metrics
)

print(result_row)

{'province': 'Ha_Noi', 'forecast_horizon': 1, 'lookback': 7, 'accuracy': 0.6641221374045801, 'precision': 0.8421052631578947, 'recall': 0.6666666666666666, 'f1': 0.7441860465116279, 'roc_auc': 0.7092261904761904}


# SECTION 10 — LOOP LOOKBACKS

In [71]:
province_results = []

trained_models = {}
for forecast_horizon in FORECAST_HORIZONS:

    print("\n")
    print("#" * 70)
    print(
        f"FORECAST HORIZON = "
        f"{forecast_horizon}"
    )
    print("#" * 70)
    for lookback in LOOKBACKS:

        print("\n")
        print("=" * 60)
        print(f"LOOKBACK = {lookback}")
        print("=" * 60)

        (
            X_train,
            X_test,
            y_train,
            y_test,
            train_df,
            test_df
        ) = prepare_train_test_data(
            df=df,
            lookback=lookback,
            forecast_horizon=forecast_horizon
        )

        model = Pipeline([

            (
                "scaler",
                StandardScaler()
            ),

            (
                "classifier",
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced"
                )
            )
        ])

        model.fit(
            X_train,
            y_train
        )

        y_pred = model.predict(
            X_test
        )

        y_prob = model.predict_proba(
            X_test
        )[:, 1]

        metrics = evaluate_model(
            y_test,
            y_pred,
            y_prob
        )

        result_row = create_result_row(
            province=province_name,
            lookback=lookback,
            forecast_horizon=forecast_horizon,
            metrics=metrics
        )

        province_results.append(
            result_row
        )

        trained_models[
            (
                lookback,
                forecast_horizon
            )
        ] = {
            "model": model,
            "metrics": metrics
        }




######################################################################
FORECAST HORIZON = 7
######################################################################


LOOKBACK = 3


LOOKBACK = 5




LOOKBACK = 7


LOOKBACK = 10


LOOKBACK = 15


LOOKBACK = 30


# SECTION 11 — RANK MODELS

In [72]:
province_results_df = (
    create_results_dataframe(
        province_results
    )
)

ranked_df = rank_models(
    province_results_df
)

ranked_df.head(3)

,province,forecast_horizon,lookback,accuracy,precision,recall,f1,roc_auc
0,Ha_Noi,7,30,0.440,0.711538,0.402174,0.513889,0.446311
1,Ha_Noi,7,15,0.368,0.632653,0.336957,0.439716,0.384058
2,Ha_Noi,7,10,0.352,0.607843,0.336957,0.433566,0.374506


# SECTION 12 — SAVE TOP MODELS

In [73]:
# ======================================
# GET TOP 3
# ======================================

top_3_models = ranked_df.head(3)

for rank, (_, row) in enumerate(
    top_3_models.iterrows(),
    start=1
):

    # ==================================
    # GET LOOKBACK
    # ==================================

    lookback = int(
        row["lookback"]
    )

    # ==================================
    # GET FORECAST HORIZON
    # ==================================

    forecast_horizon = int(
        row["forecast_horizon"]
    )

    # ==================================
    # LOAD TRAINED MODEL
    # ==================================

    model = trained_models[
        (
            lookback,
            forecast_horizon
        )
    ]["model"]

    # ==================================
    # LOAD METRICS
    # ==================================

    metrics = trained_models[
        (
            lookback,
            forecast_horizon
        )
    ]["metrics"]

    # ==================================
    # CREATE METADATA
    # ==================================

    metadata = {

        "province":
            province_name,

        "model_rank":
            rank,

        "algorithm":
            "LogisticRegression",

        "lookback":
            lookback,

        "forecast_horizon":
            forecast_horizon,

        "accuracy":
            float(metrics["accuracy"]),

        "precision":
            float(metrics["precision"]),

        "recall":
            float(metrics["recall"]),

        "f1":
            float(metrics["f1"]),

        "roc_auc":
            float(metrics["roc_auc"])
    }

    # ==================================
    # SAVE MODEL PACKAGE
    # ==================================

    save_complete_model_package(

        algorithm_name=
            "LogisticRegression",

        province_name=
            province_name,

        forecast_horizon=
            forecast_horizon,

        model_rank=
            rank,

        model=
            model,

        metadata=
            metadata
    )

    print(
        f"\nSaved TOP {rank} | "
        f"Lookback={lookback} | "
        f"Horizon={forecast_horizon}"
    )


Saved model to:
D:\Bách khoa\Intelligence System\Weather Forecast Dashboard UI\backend\ml_pipeline\LogisticRegression\models\Ha_Noi\horizon_7\model_1

Saved TOP 1 | Lookback=30 | Horizon=7

Saved model to:
D:\Bách khoa\Intelligence System\Weather Forecast Dashboard UI\backend\ml_pipeline\LogisticRegression\models\Ha_Noi\horizon_7\model_2

Saved TOP 2 | Lookback=15 | Horizon=7

Saved model to:
D:\Bách khoa\Intelligence System\Weather Forecast Dashboard UI\backend\ml_pipeline\LogisticRegression\models\Ha_Noi\horizon_7\model_3

Saved TOP 3 | Lookback=10 | Horizon=7


# SECTION 13 — LOOP ALL PROVINCES

In [74]:
ALL_RESULTS = []

for province_name in DATA_PROVINCES:

    print("\n")
    print("#" * 70)
    print(f"PROVINCE: {province_name}")
    print("#" * 70)

    # ======================================
    # LOAD DATA
    # ======================================

    df = load_province_data(
        province_name
    )

    province_results = []

    trained_models = {}

    # ======================================
    # LOOP FORECAST HORIZONS
    # ======================================

    for forecast_horizon in FORECAST_HORIZONS:

        print("\n")
        print("#" * 60)
        print(
            f"FORECAST HORIZON = "
            f"{forecast_horizon}"
        )
        print("#" * 60)

        # ==================================
        # LOOP LOOKBACKS
        # ==================================

        for lookback in LOOKBACKS:

            print("\n")
            print("=" * 60)
            print(f"LOOKBACK = {lookback}")
            print("=" * 60)

            (
                X_train,
                X_test,
                y_train,
                y_test,
                train_df,
                test_df
            ) = prepare_train_test_data(
                df=df,
                lookback=lookback,
                forecast_horizon=forecast_horizon
            )

            # ==============================
            # CREATE MODEL
            # ==============================

            model = Pipeline([

                (
                    "scaler",
                    StandardScaler()
                ),

                (
                    "classifier",
                    LogisticRegression(
                        max_iter=1000,
                        class_weight="balanced"
                    )
                )
            ])

            # ==============================
            # TRAIN
            # ==============================

            model.fit(
                X_train,
                y_train
            )

            # ==============================
            # PREDICT
            # ==============================

            y_pred = model.predict(
                X_test
            )

            y_prob = model.predict_proba(
                X_test
            )[:, 1]

            # ==============================
            # EVALUATE
            # ==============================

            metrics = evaluate_model(
                y_test,
                y_pred,
                y_prob
            )

            # ==============================
            # RESULT ROW
            # ==============================

            result_row = create_result_row(
                province=province_name,
                lookback=lookback,
                forecast_horizon=forecast_horizon,
                metrics=metrics
            )

            province_results.append(
                result_row
            )

            ALL_RESULTS.append(
                result_row
            )

            # ==============================
            # STORE MODEL
            # ==============================

            trained_models[
                (
                    lookback,
                    forecast_horizon
                )
            ] = {

                "model":
                    model,

                "metrics":
                    metrics
            }

    # ======================================
    # CREATE RESULTS DATAFRAME
    # ======================================

    province_results_df = (
        create_results_dataframe(
            province_results
        )
    )

    # ======================================
    # RANK MODELS
    # ======================================

    ranked_df = rank_models(
        province_results_df
    )

    print("\n")
    print("=" * 60)
    print("TOP 3 MODELS")
    print("=" * 60)

    print(
        ranked_df.head(3)
    )

    # ======================================
    # SAVE TOP 3 MODELS
    # ======================================

    top_3_models = ranked_df.head(3)

    for rank, (_, row) in enumerate(
        top_3_models.iterrows(),
        start=1
    ):

        lookback = int(
            row["lookback"]
        )

        forecast_horizon = int(
            row["forecast_horizon"]
        )

        # ==================================
        # LOAD MODEL
        # ==================================

        model = trained_models[
            (
                lookback,
                forecast_horizon
            )
        ]["model"]

        # ==================================
        # LOAD METRICS
        # ==================================

        metrics = trained_models[
            (
                lookback,
                forecast_horizon
            )
        ]["metrics"]

        # ==================================
        # METADATA
        # ==================================

        metadata = {

            "province":
                province_name,

            "model_rank":
                rank,

            "algorithm":
                "LogisticRegression",

            "lookback":
                lookback,

            "forecast_horizon":
                forecast_horizon,

            "accuracy":
                float(metrics["accuracy"]),

            "precision":
                float(metrics["precision"]),

            "recall":
                float(metrics["recall"]),

            "f1":
                float(metrics["f1"]),

            "roc_auc":
                float(metrics["roc_auc"])
        }

        # ==================================
        # SAVE MODEL
        # ==================================

        save_complete_model_package(

            algorithm_name=
                "LogisticRegression",

            province_name=
                province_name,

            forecast_horizon=
                forecast_horizon,

            model_rank=
                rank,

            model=
                model,

            metadata=
                metadata
        )

        print(
            f"\nSaved TOP {rank} | "
            f"Lookback={lookback} | "
            f"Horizon={forecast_horizon}"
        )

    print("\n")
    print("=" * 60)
    print(f"FINISHED: {province_name}")
    print("=" * 60)



######################################################################
PROVINCE: An_Giang
######################################################################


############################################################
FORECAST HORIZON = 7
############################################################


LOOKBACK = 3


LOOKBACK = 5


LOOKBACK = 7


LOOKBACK = 10


LOOKBACK = 15


LOOKBACK = 30


TOP 3 MODELS
   province  forecast_horizon  lookback  accuracy  precision    recall  \
0  An_Giang                 7        15     0.624   0.814815  0.343750   
1  An_Giang                 7        30     0.552   0.605263  0.359375   
2  An_Giang                 7         5     0.624   0.904762  0.296875   

         f1   roc_auc  
0  0.483516  0.653689  
1  0.450980  0.563012  
2  0.447059  0.665727  

Saved model to:
D:\Bách khoa\Intelligence System\Weather Forecast Dashboard UI\backend\ml_pipeline\LogisticRegression\models\An_Giang\horizon_7\model_1

Saved TOP 1 | Lookback=15 | Horizon=7

# SECTION 14 — SAVE FINAL CSV

In [75]:
# ======================================
# CREATE FINAL RESULTS DATAFRAME
# ======================================

all_results_df = create_results_dataframe(
    ALL_RESULTS
)

# ======================================
# SORT RESULTS
# ======================================

all_results_df = all_results_df.sort_values(

    by=[
        "province",
        "forecast_horizon",
        "f1"
    ],

    ascending=[
        True,
        True,
        False
    ]
)

# ======================================
# CREATE SAVE DIRECTORY
# ======================================

save_path = Path(
    "results"
)

save_path.mkdir(
    parents=True,
    exist_ok=True
)

# ======================================
# SAVE CSV
# ======================================

csv_path = (
    save_path /
    f"logistic_regression_results_{FORECAST_HORIZONS[0]}.csv"
)

all_results_df.to_csv(
    csv_path,
    index=False
)

# ======================================
# PRINT SUMMARY
# ======================================

print("\n")
print("=" * 60)
print("TRAINING COMPLETED")
print("=" * 60)

print(
    f"Total Results: "
    f"{len(all_results_df)}"
)

print(
    f"Saved CSV:\n"
    f"{csv_path}"
)

print("\nTop Rows:")
print(
    all_results_df.head()
)



TRAINING COMPLETED
Total Results: 204
Saved CSV:
results\logistic_regression_results_7.csv

Top Rows:
   province  forecast_horizon  lookback  accuracy  precision    recall  \
4  An_Giang                 7        15     0.624   0.814815  0.343750   
5  An_Giang                 7        30     0.552   0.605263  0.359375   
1  An_Giang                 7         5     0.624   0.904762  0.296875   
0  An_Giang                 7         3     0.616   0.944444  0.265625   
2  An_Giang                 7         7     0.568   0.812500  0.203125   

         f1   roc_auc  
4  0.483516  0.653689  
5  0.450980  0.563012  
1  0.447059  0.665727  
0  0.414634  0.685963  
2  0.325000  0.678535  
